# OLMo-2-1B Two-Stage Concentration Experiment

**Stage 1 (sweep):** Fine-tune at varying code concentrations  
- `c` = fraction of CODE data in batch. `c=1.0` = pure code, `c=0.5` = half code + half pretrain
- Each `(c, lr, seed)` produces a separate fine-tuned model

**Stage 2 (fixed):** 100% pretraining data for all runs  
- Measures how fast each stage-1 model **forgets code** when retrained on pretraining data
- Code val loss should rise (forgetting), pretrain val loss should fall (recovery)

**Key question:** Does mixing pretraining data during fine-tuning (stage 1) produce models that are more robust to subsequent forgetting (stage 2)?

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

RESULTS_DIR = Path("results")
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path("olmo2/results")

CONC_COLORS = {
    0.0: '#1b9e77', 0.1: '#377eb8', 0.3: '#4daf4a', 0.5: '#ff7f00',
    0.7: '#984ea3', 0.9: '#e41a1c', 1.0: '#a65628',
}

def parse_s2_name(name):
    """Parse 'stage2_c0.5_lr5e-05_s2lr5e-05_s42' -> (c, lr, seed)."""
    parts = name.replace('stage2_', '').split('_')
    c = float(parts[0][1:])
    lr = float(parts[1][2:])
    # seed is last part starting with 's' (not 's2')
    for p in reversed(parts):
        if p.startswith('s') and not p.startswith('s2'):
            try:
                seed = int(p[1:])
                return c, lr, seed
            except ValueError:
                continue
    return c, lr, int(parts[-1][1:])

print(f"Results dir: {RESULTS_DIR.resolve()}")

## 1. Load data

In [ ]:
# Stage 1 runs (one per concentration/lr/seed)
s1_runs = {}
for d in sorted(RESULTS_DIR.iterdir()):
    if not d.is_dir() or not d.name.startswith('stage1_c'):
        continue
    csv_path = d / 'scalar_metrics.csv'
    if not csv_path.exists():
        continue
    try:
        parts = d.name.replace('stage1_', '').split('_')
        c = float(parts[0][1:])
        lr = float(parts[1][2:])
        seed = int(parts[2][1:])
    except:
        continue
    df = pd.read_csv(csv_path)
    df['concentration'] = c
    df['lr_val'] = lr
    df['seed'] = seed
    s1_runs[d.name] = df

print(f"Stage 1: {len(s1_runs)} runs")
for name in sorted(s1_runs):
    df = s1_runs[name]
    print(f"  {name}: {len(df)} measurements, steps 0..{df['step'].max()}")

# Stage 2 runs
s2_runs = {}
configs = {}
for d in sorted(RESULTS_DIR.iterdir()):
    if not d.is_dir() or not d.name.startswith('stage2_c'):
        continue
    csv_path = d / 'scalar_metrics.csv'
    if not csv_path.exists():
        continue
    try:
        c, lr, seed = parse_s2_name(d.name)
    except:
        continue
    df = pd.read_csv(csv_path)
    df['concentration'] = c
    df['lr_val'] = lr
    df['seed'] = seed
    s2_runs[d.name] = df
    cfg_path = d / 'config.json'
    if cfg_path.exists():
        with open(cfg_path) as f:
            configs[d.name] = json.load(f)

print(f"Stage 2: {len(s2_runs)} runs")
for name in sorted(s2_runs):
    df = s2_runs[name]
    print(f"  {name}: {len(df)} measurements, steps 0..{df['step'].max()}")

all_s2 = pd.concat(s2_runs.values(), ignore_index=True) if s2_runs else pd.DataFrame()

## 2. Stage 1: Code fine-tuning at varying concentrations

Each line = one concentration level. Higher `c` = more code in training mix.
Code val loss should decrease faster at higher concentrations.

In [ ]:
if s1_runs:
    all_s1 = pd.concat(s1_runs.values(), ignore_index=True)
    lrs = sorted(all_s1['lr_val'].unique())
    for lr in lrs:
        sub = all_s1[all_s1['lr_val'] == lr]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        for c in sorted(sub['concentration'].unique()):
            csub = sub[sub['concentration'] == c]
            grouped = csub.groupby('step')
            color = CONC_COLORS.get(c, '#333')
            ax1.plot(grouped['code_val_loss'].mean().index, grouped['code_val_loss'].mean().values, label=f'c={c}', color=color, lw=2)
            ax2.plot(grouped['pretrain_val_loss'].mean().index, grouped['pretrain_val_loss'].mean().values, label=f'c={c}', color=color, lw=2)
        ax1.set_xlabel('Step'); ax1.set_ylabel('Code Val Loss')
        ax1.set_title(f'Stage 1: Code Learning (lr={lr})'); ax1.legend()
        ax2.set_xlabel('Step'); ax2.set_ylabel('Pretrain Val Loss')
        ax2.set_title(f'Stage 1: Pretrain Drift (lr={lr})'); ax2.legend()
        plt.tight_layout(); plt.show()
else:
    print('No stage 1 data found')

## 3. Stage 2: Code forgetting (100% pretraining)

All stage-2 runs use 100% pretraining data. Lines are grouped by the stage-1 concentration `c`.
Higher stage-1 `c` (more code during fine-tuning) should mean more code to forget.

In [ ]:
def plot_s2_metric(metric, ylabel, title, lr_filter=None):
    df = all_s2.copy()
    if lr_filter is not None:
        df = df[df['lr_val'] == lr_filter]
    lrs = [lr_filter] if lr_filter else sorted(df['lr_val'].unique())
    
    for lr in lrs:
        sub = df[df['lr_val'] == lr]
        fig, ax = plt.subplots(figsize=(10, 6))
        for c in sorted(sub['concentration'].unique()):
            csub = sub[sub['concentration'] == c]
            grouped = csub.groupby('step')[metric]
            mean, std = grouped.mean(), grouped.std().fillna(0)
            color = CONC_COLORS.get(c, '#333')
            ax.plot(mean.index, mean.values, label=f'c={c}', color=color, lw=2)
            if csub['seed'].nunique() > 1:
                ax.fill_between(mean.index, mean - std, mean + std, alpha=0.15, color=color)
        ax.set_xlabel('Stage 2 Step'); ax.set_ylabel(ylabel)
        ax.set_title(f'{title} (stage-1 lr={lr})')
        ax.legend()
        plt.tight_layout(); plt.show()

if len(all_s2) > 0:
    plot_s2_metric('code_val_loss', 'Code Val Loss', 'Code Forgetting (stage 2 = 100% pretrain)')
else:
    print('No stage 2 data')

## 4. Pretraining recovery (stage 2)

Pretrain val loss during stage 2 -- should decrease as the model re-learns general text.

In [ ]:
if len(all_s2) > 0:
    plot_s2_metric('pretrain_val_loss', 'Pretrain Val Loss', 'Pretraining Recovery (stage 2 = 100% pretrain)')
else:
    print('No stage 2 data')

## 5. Gradient cosine similarity (code vs pretrain)

In [ ]:
if len(all_s2) > 0 and 'grad_cossim_whole_model' in all_s2.columns:
    plot_s2_metric('grad_cossim_whole_model', 'Gradient CosSim', 'Gradient Conflict (code vs pretrain)')
else:
    print('No gradient cossim data')

## 6. Pareto trade-off

Each point = one (stage-1 concentration, lr) condition at end of stage 2.
X = final pretrain loss, Y = final code loss.

In [ ]:
LR_MARKERS = {1e-5: 'o', 5e-5: 's', 1e-4: '^'}

if s2_runs:
    fig, ax = plt.subplots(figsize=(10, 8))
    for name, df in s2_runs.items():
        c, lr, seed = parse_s2_name(name)
        last = df.iloc[-1]
        ax.scatter(last['pretrain_val_loss'], last['code_val_loss'],
                   color=CONC_COLORS.get(c, '#333'), marker=LR_MARKERS.get(lr, 'o'),
                   s=100, edgecolors='black', lw=0.5, zorder=5)

    for c, color in sorted(CONC_COLORS.items()):
        if c in all_s2['concentration'].unique():
            ax.scatter([], [], color=color, s=80, label=f'c={c}')
    for lr, marker in sorted(LR_MARKERS.items()):
        if lr in all_s2['lr_val'].unique():
            ax.scatter([], [], color='gray', marker=marker, s=80, label=f'lr={lr}')

    ax.set_xlabel('Final Pretrain Val Loss (lower = better recovery)')
    ax.set_ylabel('Final Code Val Loss (lower = less forgetting)')
    ax.set_title('Trade-off: Pretraining Recovery vs Code Retention (after stage 2)')
    ax.legend(ncol=2)
    plt.tight_layout(); plt.show()
else:
    print('No data')

## 7. Learning rate effect at c=1.0 (pure pretraining)

In [ ]:
c1_df = all_s2[all_s2['concentration'] == 1.0] if len(all_s2) > 0 else pd.DataFrame()

if len(c1_df) > 0:
    colors = ['#e41a1c', '#377eb8', '#4daf4a']
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    for i, lr in enumerate(sorted(c1_df['lr_val'].unique())):
        sub = c1_df[c1_df['lr_val'] == lr]
        for ax, metric, title in [(ax1, 'code_val_loss', 'Code Forgetting'), (ax2, 'pretrain_val_loss', 'Pretrain Recovery')]:
            grouped = sub.groupby('step')[metric]
            mean, std = grouped.mean(), grouped.std().fillna(0)
            ax.plot(mean.index, mean.values, label=f'lr={lr}', color=colors[i % 3], lw=2)
            if sub['seed'].nunique() > 1:
                ax.fill_between(mean.index, mean - std, mean + std, alpha=0.15, color=colors[i % 3])
    ax1.set_title('Code Val Loss (stage-1 c=1.0)'); ax2.set_title('Pretrain Val Loss (stage-1 c=1.0)')
    for ax in (ax1, ax2): ax.set_xlabel('Stage 2 Step'); ax.legend()
    plt.tight_layout(); plt.show()
else:
    print('No c=1.0 runs')

## 8. Per-layer gradient cossim heatmap

In [ ]:
# Pick a run to visualize
heatmap_run = None
for name in sorted(s2_runs.keys()):
    layer_csv = RESULTS_DIR / name / 'metrics.csv'
    if layer_csv.exists():
        heatmap_run = name
        break

if heatmap_run:
    layer_df = pd.read_csv(RESULTS_DIR / heatmap_run / 'metrics.csv')
    cossim_col = 'grad_cossim_code_vs_pretrain'
    if cossim_col in layer_df.columns:
        steps = sorted(layer_df['step'].unique())
        layers = sorted(layer_df['layer'].unique())
        mat = np.full((len(layers), len(steps)), np.nan)
        for si, s in enumerate(steps):
            sub = layer_df[layer_df['step'] == s].set_index('layer')
            for li, la in enumerate(layers):
                if la in sub.index:
                    val = sub.loc[la, cossim_col]
                    mat[li, si] = val.iloc[0] if hasattr(val, 'iloc') else val
        fig, ax = plt.subplots(figsize=(16, 8))
        im = ax.imshow(mat, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_yticks(range(len(layers))); ax.set_yticklabels(layers, fontsize=7)
        tick_idx = list(range(0, len(steps), max(1, len(steps) // 12)))
        ax.set_xticks(tick_idx); ax.set_xticklabels([steps[i] for i in tick_idx], fontsize=8, rotation=45)
        ax.set_xlabel('Step'); ax.set_ylabel('Layer')
        ax.set_title(f'Per-Layer Gradient CosSim -- {heatmap_run}')
        fig.colorbar(im, ax=ax, label='Cosine Similarity')
        plt.tight_layout(); plt.show()
    else:
        print(f'No {cossim_col} in layer metrics')
else:
    print('No per-layer metrics found')

## 9. Summary table

In [ ]:
if s2_runs:
    rows = []
    for name, df in s2_runs.items():
        c, lr, seed = parse_s2_name(name)
        last = df.iloc[-1]
        rows.append({
            'c': c, 'lr': lr, 'seed': seed, 'steps': int(last['step']),
            'code_val': last['code_val_loss'], 'pretrain_val': last['pretrain_val_loss'],
            'cossim': last.get('grad_cossim_whole_model', np.nan),
        })
    summary = pd.DataFrame(rows).sort_values(['c', 'lr', 'seed'])
    display(summary)
    
    print('\n--- Averaged over seeds ---')
    agg = summary.groupby(['c', 'lr']).agg(
        code_mean=('code_val', 'mean'), code_std=('code_val', 'std'),
        pretrain_mean=('pretrain_val', 'mean'), pretrain_std=('pretrain_val', 'std'),
        cossim_mean=('cossim', 'mean'), n=('seed', 'count'),
    ).reset_index()
    display(agg)
else:
    print('No data')

## 10. Benchmark results (if available)

In [ ]:
bench_data = []
for d in sorted(RESULTS_DIR.iterdir()):
    bench_file = d / 'all_benchmark_results.json'
    if not bench_file.exists():
        continue
    try:
        c, lr, seed = parse_s2_name(d.name)
    except:
        continue
    with open(bench_file) as f:
        data = json.load(f)
    for entry in data:
        row = {'c': c, 'lr': lr, 'seed': seed, 'step': entry.get('step', -1)}
        row.update(entry.get('scores', {}))
        bench_data.append(row)

if bench_data:
    bench_df = pd.DataFrame(bench_data)
    display(bench_df)
    
    final_bench = bench_df.loc[bench_df.groupby(['c', 'lr', 'seed'])['step'].idxmax()]
    score_cols = [col for col in final_bench.columns if col not in ('c', 'lr', 'seed', 'step')]
    
    if score_cols:
        for lr in sorted(final_bench['lr'].unique()):
            sub = final_bench[final_bench['lr'] == lr]
            fig, ax = plt.subplots(figsize=(10, 6))
            for col in score_cols:
                grouped = sub.groupby('c')[col]
                mean = grouped.mean()
                std = grouped.std().fillna(0)
                ax.errorbar(mean.index, mean.values, yerr=std.values, marker='o', label=col, capsize=4, lw=2)
            ax.set_xlabel('Stage-1 Concentration (code fraction)')
            ax.set_ylabel('Score')
            ax.set_title(f'Benchmarks after Stage 2 (stage-1 lr={lr})')
            ax.legend(fontsize=9)
            plt.tight_layout(); plt.show()
else:
    print('No benchmark results yet. Run eval_benchmarks.py on your checkpoints.')

## 11. Single-run inspector

Change `RUN_NAME` to zoom into any specific run.

In [ ]:
RUN_NAME = sorted(s2_runs.keys())[0] if s2_runs else None

if RUN_NAME and RUN_NAME in s2_runs:
    df = s2_runs[RUN_NAME]
    print(f"Inspecting: {RUN_NAME}")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].plot(df['step'], df['train_loss'], label='train', color='#999', ls='--')
    axes[0, 0].plot(df['step'], df['code_val_loss'], label='code val', color='#377eb8', lw=2)
    axes[0, 0].plot(df['step'], df['pretrain_val_loss'], label='pretrain val', color='#e41a1c', lw=2)
    axes[0, 0].set_title('Losses'); axes[0, 0].legend(); axes[0, 0].set_xlabel('Step')
    
    if 'grad_cossim_whole_model' in df.columns:
        axes[0, 1].plot(df['step'], df['grad_cossim_whole_model'], color='#984ea3', lw=2)
        axes[0, 1].axhline(0, color='gray', ls='--', lw=0.5)
        axes[0, 1].set_title('Gradient CosSim'); axes[0, 1].set_xlabel('Step')
    
    if 'avg_weight_drift_rel' in df.columns:
        axes[1, 0].plot(df['step'], df['avg_weight_drift_rel'], color='#ff7f00', lw=2)
        axes[1, 0].set_title('Weight Drift'); axes[1, 0].set_xlabel('Step')
    
    if 'lr' in df.columns:
        axes[1, 1].plot(df['step'], df['lr'], color='#a65628', lw=2)
        axes[1, 1].set_title('Learning Rate'); axes[1, 1].set_xlabel('Step')
    
    plt.suptitle(RUN_NAME, fontsize=14)
    plt.tight_layout(); plt.show()
    
    if RUN_NAME in configs:
        print('\nConfig:'); [print(f'  {k}: {v}') for k, v in configs[RUN_NAME].items()]
else:
    print('No runs loaded.')